In [2]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import re
import string 
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np


In [3]:
df = pd.read_csv("IMDB.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace(':', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Removing URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f"Error during text normalization: {e}")
        raise

<>:32: SyntaxWarning: invalid escape sequence '\s'
<>:32: SyntaxWarning: invalid escape sequence '\s'
C:\Users\ASUS\AppData\Local\Temp\ipykernel_19640\2307984544.py:32: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [5]:
df = normalize_text(df)
df.head()

,review,sentiment
0,one reviewer mentioned watching oz episode hoo...,positive
1,wonderful little production br br the filming ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically there s family little boy jake think...,negative
4,petter mattei s love time money visually stunn...,positive


In [6]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [7]:
x = df['sentiment'].isin(['positive', 'negative'])
df = df[x]

In [8]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head

<bound method NDFrame.head of                                                   review  sentiment
0      one reviewer mentioned watching oz episode hoo...          1
1      wonderful little production br br the filming ...          1
2      thought wonderful way spend time hot summer we...          1
3      basically there s family little boy jake think...          0
4      petter mattei s love time money visually stunn...          1
...                                                  ...        ...
49995  thought movie right good job creative original...          1
49996  bad plot bad dialogue bad acting idiotic direc...          0
49997  catholic taught parochial elementary school nu...          0
49998  going disagree previous comment side maltin on...          0
49999  one expects star trek movie high art fan expec...          0

[50000 rows x 2 columns]>

In [9]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [10]:
vectorizer = CountVectorizer(max_features = 50)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [11]:
# train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [12]:
import dagshub
mlflow.set_tracking_uri('https://dagshub.com/AmitKr-06/IMDB-movie-predictor.mlflow')
dagshub.init(repo_owner='AmitKr-06', repo_name='IMDB-movie-predictor', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")

Accessing as AmitKr-06

Initialized MLflow to track repo "AmitKr-06/IMDB-movie-predictor"

Repository AmitKr-06/IMDB-movie-predictor initialized!

<Experiment: artifact_location='mlflow-artifacts:/024ed4e2113a4bc0914855b270aef80f', creation_time=1790001223935, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1790001223935, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [15]:
# Performing experiment
import logging
import os
import time

# Configure logging
logging.basicConfig(level = logging.INFO, format = "%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()

    try:
        logging.info("Logging Preprocessing Parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.2)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter = 1000)   # max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters....")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making Predictions....")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall",recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model....")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time 
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        notebook_path = "exp1.ipynb"
        logging.info("Execting Jupyter Notebook.")
        os.system(f"Jupyter nbconvert --to notebook ---execute --inplace {notebook_path}")
        mlflow.log_artifact(notebook_path)

        logging.info("Notebook execution and logging complete.")

        # Print the results for verificatioin
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info = True)



2026-09-21 20:42:50,535 - INFO - Starting MLflow run...
2026-09-21 20:43:13,118 - INFO - Logging Preprocessing Parameters...
2026-09-21 20:43:14,303 - INFO - Initializing Logistic Regression model...
2026-09-21 20:43:14,304 - INFO - Fitting the model...
2026-09-21 20:43:14,416 - INFO - Model training complete.
2026-09-21 20:43:14,417 - INFO - Logging model parameters....
2026-09-21 20:43:14,796 - INFO - Making Predictions....
2026-09-21 20:43:14,798 - INFO - Calculating evaluation metrics...
2026-09-21 20:43:14,810 - INFO - Logging evaluation metrics...
2026-09-21 20:43:16,520 - INFO - Saving and logging the model....
2026/09/21 20:43:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-09-21 20:43:32,788 - INFO - Model training and logging completed in 19.67 seconds.
2026-09-21 20:43:32,789 - INFO - Execting Jupyter Notebook.
2026-09-21 20:43:38,748 - INFO - Notebook execution and logging complete.
2026-09-21 20:43:38,749 - INFO - Accuracy: 0.

🏃 View run silent-lark-25 at: https://dagshub.com/AmitKr-06/IMDB-movie-predictor.mlflow/#/experiments/0/runs/383539963368491b8d7f6c431f3a9174
🧪 View experiment at: https://dagshub.com/AmitKr-06/IMDB-movie-predictor.mlflow/#/experiments/0
